# RF-DETR 74-Class Hidden45 Training Notebook

## Goal

Train RF-DETR on the 74-class pill dataset built from:

- base 56 classes, filled to at least 45 annotations across train+valid
- hidden N57-N74 crop classes
- original downloaded competition test images for inference-only `test` split

The notebook is a runnable handoff wrapper around the scripts in this folder. The default config is tuned for local Apple Silicon MPS. By default it prepares and validates the dataset, then runs a dry-run. Set `RUN_FULL_TRAIN = True` in the parameter cell to start real training.

## Setup

Use this notebook from the team repo. If opened from the repo root, it will move into `RF_DETR_split_ver` automatically.

Key assumptions:

- `rfdetr[train,loggers]` is installed in the active Python kernel.
- The prepared 56-class and hidden N18 source folders exist locally or are restored from Drive.
- Test images come from the original downloaded competition `test_images` folder and have no public annotations.
- Local MPS defaults use `batch_size=1`, `grad_accum_steps=16`, `num_workers=2`, `pin_memory=false`, and `multi_scale=false` for stable memory use.

In [28]:
# Parameters
from pathlib import Path

RUN_INSTALL_DEPS = False      # Set True only in a fresh Colab/kernel.
RUN_PREPARE_DATASET = True    # Rebuild RF-DETR train/valid/test folder.
RUN_DRY_RUN = True            # Validate config without launching long training.
RUN_FULL_TRAIN = False       # Keep False in the committed notebook; set True locally to train/resume.
RUN_FINALIZE_ONLY = False    # Set True after an existing run finishes to copy best_total and best mAP75 checkpoints only.
TRAIN_EPOCHS_OVERRIDE = None  # Keep None for config-controlled epochs; use a small number only for debug runs.

# Evaluation controls. Full validation inference is off by default because it re-runs the model over valid images.
RUN_VALIDATION_INFERENCE = False
VALIDATION_SCORE_THR = 0.001
VALIDATION_MAX_DET_PER_IMAGE = 100  # Match the RT-DETR/COCO default maxDets=100 unless the competition rule changes.
VALIDATION_MAX_IMAGES = None        # Example: 20 for a quick smoke eval, None for full valid split.

# Local defaults used by the current project workspace.
BASE56_DIR = Path('/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/train_56_45_merged_coco')
HIDDEN18_DIR = Path('/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/hidden_train_import')
TEST_IMAGES_DIR = Path('/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/sprint_ai_project1_data/test_images')
RFDETR_DATASET_DIR = Path('/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45')
CONFIG_PATH = Path('config_74_hidden45.yaml')

LINK_MODE = 'symlink'  # symlink avoids duplicate image copies. Use copy for portable upload folders.
DEVICE_NOTE = 'Local MPS defaults are active. For Colab/GPU, set train.device=cuda and train.pin_memory=true.'
print(DEVICE_NOTE)

Local MPS defaults are active. For Colab/GPU, set train.device=cuda and train.pin_memory=true.


In [29]:
# Imports and working directory
import json
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

start_cwd = Path.cwd()
if (start_cwd / 'RF_DETR_split_ver').exists():
    os.chdir(start_cwd / 'RF_DETR_split_ver')
elif not (start_cwd / 'prepare_74_hidden45_dataset.py').exists():
    raise RuntimeError(f'Open this notebook from repo root or RF_DETR_split_ver, not {start_cwd}')

WORK_DIR = Path.cwd()
print('WORK_DIR:', WORK_DIR)
print('Python:', sys.executable)
print('Config:', (WORK_DIR / CONFIG_PATH).resolve())

WORK_DIR: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver
Python: /Users/pio/.DataAnalysis/bin/python
Config: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver/config_74_hidden45.yaml


In [30]:
# Optional dependency install
if RUN_INSTALL_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK_DIR.parent / 'requirements.txt')], check=True)
else:
    print('Skipping dependency install. Set RUN_INSTALL_DEPS=True in a fresh environment.')

Skipping dependency install. Set RUN_INSTALL_DEPS=True in a fresh environment.


In [31]:
# Environment check
import torch

try:
    import rfdetr
    rfdetr_status = 'ok'
except Exception as exc:
    rfdetr_status = f'IMPORT FAILED: {exc}'

env_summary = {
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'mps_available': bool(getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()),
    'rfdetr': rfdetr_status,
}
print(json.dumps(env_summary, indent=2, ensure_ascii=False))

{
  "torch_version": "2.12.0",
  "cuda_available": false,
  "mps_available": true,
  "rfdetr": "ok"
}


## Data

This section validates source folders and prepares the RF-DETR directory layout:

```text
rfdetr_dataset_74_hidden45/
  train/_annotations.coco.json
  valid/_annotations.coco.json
  test/_annotations.coco.json
```

`train` and `valid` contain labeled images. `test` contains the original downloaded test images with an empty annotation list.

In [32]:
# Validate source paths
paths = {
    'BASE56_DIR': BASE56_DIR,
    'HIDDEN18_DIR': HIDDEN18_DIR,
    'TEST_IMAGES_DIR': TEST_IMAGES_DIR,
}
missing = {name: str(path) for name, path in paths.items() if not path.exists()}
if missing:
    raise FileNotFoundError(json.dumps(missing, indent=2, ensure_ascii=False))

for name, path in paths.items():
    print(f'{name}: {path}')

BASE56_DIR: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/train_56_45_merged_coco
HIDDEN18_DIR: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/hidden_train_import
TEST_IMAGES_DIR: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/sprint_ai_project1_data/test_images


In [33]:
# Prepare RF-DETR dataset
if RUN_PREPARE_DATASET:
    prepare_cmd = [
        sys.executable,
        'prepare_74_hidden45_dataset.py',
        '--base56-dir', str(BASE56_DIR),
        '--hidden18-dir', str(HIDDEN18_DIR),
        '--test-images-dir', str(TEST_IMAGES_DIR),
        '--out-dir', str(RFDETR_DATASET_DIR),
        '--link-mode', LINK_MODE,
    ]
    print('run:', ' '.join(prepare_cmd))
    subprocess.run(prepare_cmd, check=True)
else:
    print('Skipping dataset preparation. Using existing:', RFDETR_DATASET_DIR)

run: /Users/pio/.DataAnalysis/bin/python prepare_74_hidden45_dataset.py --base56-dir /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/train_56_45_merged_coco --hidden18-dir /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/hidden_train_import --test-images-dir /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/sprint_ai_project1_data/test_images --out-dir /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45 --link-mode symlink
{
  "base56_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/train_56_45_merged_coco",
  "hidden18_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/hidden_train_import",
  "test_images_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/sprint_ai_project1_data/test_images",
  "out_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45",
  "link_

In [34]:
# Load dataset summary
summary_path = RFDETR_DATASET_DIR / 'summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
display(pd.DataFrame(summary['splits']))
print('category_id_semantics:', summary['category_id_semantics'])

,split,source_split,images,annotations,categories,base56_images,hidden18_images,test_images_dir
0,train,train,3502,3965,74,1875.0,1627.0,NaN
1,valid,val,376,452,74,218.0,158.0,NaN
2,test,original_test,842,0,74,NaN,NaN,/Users/pio/Documents/AIENGINEERCOURSE/detectio...


category_id_semantics: COCO category_id is the numeric K-code. RF-DETR remaps sparse ids internally.


In [35]:
# Sanity checks: split sizes, K-code ids, bbox bounds, and 45-fill coverage
def load_split(split: str) -> dict:
    return json.loads((RFDETR_DATASET_DIR / split / '_annotations.coco.json').read_text(encoding='utf-8'))

splits = {split: load_split(split) for split in ['train', 'valid', 'test']}
rows = []
for split, data in splits.items():
    category_ids = sorted(int(category['id']) for category in data['categories'])
    annotation_ids = sorted({int(annotation['category_id']) for annotation in data['annotations']})
    file_count = sum(1 for path in (RFDETR_DATASET_DIR / split).iterdir() if path.name != '_annotations.coco.json')
    rows.append({
        'split': split,
        'json_images': len(data['images']),
        'files': file_count,
        'annotations': len(data['annotations']),
        'categories': len(category_ids),
        'category_id_min': min(category_ids),
        'category_id_max': max(category_ids),
        'annotation_id_min': min(annotation_ids) if annotation_ids else None,
        'annotation_id_max': max(annotation_ids) if annotation_ids else None,
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

assert summary_df.loc[summary_df['split'].eq('train'), 'categories'].item() == 74
assert summary_df.loc[summary_df['split'].eq('valid'), 'categories'].item() == 74
assert summary_df.loc[summary_df['split'].eq('test'), 'json_images'].item() == 842
assert summary_df.loc[summary_df['split'].eq('test'), 'annotations'].item() == 0

# Validate bbox bounds on labeled splits.
for split in ['train', 'valid']:
    data = splits[split]
    images = {image['id']: image for image in data['images']}
    for annotation in data['annotations']:
        image = images[annotation['image_id']]
        x, y, w, h = map(float, annotation['bbox'])
        assert w > 0 and h > 0, (split, annotation)
        assert x >= 0 and y >= 0, (split, image['file_name'], annotation['bbox'])
        assert x + w <= image['width'] + 1e-6, (split, image['file_name'], annotation['bbox'])
        assert y + h <= image['height'] + 1e-6, (split, image['file_name'], annotation['bbox'])

train_counts = Counter(int(annotation['category_id']) for annotation in splits['train']['annotations'])
valid_counts = Counter(int(annotation['category_id']) for annotation in splits['valid']['annotations'])
combined_counts = train_counts + valid_counts
all_category_ids = sorted(int(category['id']) for category in splits['train']['categories'])
min_combined = min(combined_counts[category_id] for category_id in all_category_ids)
classes_below_45 = [category_id for category_id in all_category_ids if combined_counts[category_id] < 45]
print('train+valid min annotations per class:', min_combined)
print('classes below 45:', classes_below_45)
assert not classes_below_45

category_mapping = pd.read_csv(RFDETR_DATASET_DIR / 'category_mapping.csv')
display(category_mapping.head())
display(category_mapping.tail())

,split,json_images,files,annotations,categories,category_id_min,category_id_max,annotation_id_min,annotation_id_max
0,train,3502,3502,3965,74,1900,44199,1900.0,44199.0
1,valid,376,376,452,74,1900,44199,1900.0,44199.0
2,test,842,842,0,74,1900,44199,NaN,NaN


train+valid min annotations per class: 45
classes below 45: []


,category_id,rfdetr_internal_label,n_number,source_dataset,source_category_id,name,mapping_code,print_front,print_back,candidate_status,decision_note
0,1900,0,N01,base56_45fill,0,보령부스파정 5mg,K-001900,BSP,5,NaN,NaN
1,2483,1,N02,base56_45fill,1,뮤테란캡슐 100mg,K-002483,Hanwha MUC100,NaN,NaN,NaN
2,3351,2,N03,base56_45fill,2,일양하이트린정 2mg,K-003351,I분할선Y,HT,NaN,NaN
3,3483,3,N04,base56_45fill,3,기넥신에프정(은행엽엑스)(수출용),K-003483,SK,G40,NaN,NaN
4,3544,4,N05,base56_45fill,4,무코스타정(레바미피드)(비매품),K-003544,OG33,NaN,NaN,NaN


,category_id,rfdetr_internal_label,n_number,source_dataset,source_category_id,name,mapping_code,print_front,print_back,candidate_status,decision_note
69,35206,69,N53,base56_45fill,52,아토젯정 10/40mg,K-035206,337,NaN,NaN,NaN
70,36637,70,N54,base56_45fill,53,로수젯정10/5밀리그램,K-036637,R5,분할선,NaN,NaN
71,38162,71,N55,base56_45fill,54,로수바미브정 10/20mg,K-038162,RE20,Y분할선H,NaN,NaN
72,41768,72,N56,base56_45fill,55,카발린캡슐 25mg,K-041768,CJ PGN 25,NaN,NaN,NaN
73,44199,73,N66,hidden18_45fill,44199,케이캡정 50mg,K-044199,K분할선50,분할선,hidden_confirmed,K|50 반복 unknown과 케이캡정 50mg exact match. K|150처...


## Training

Run a dry-run first. Then set `RUN_FULL_TRAIN = True` locally and rerun the final training cell when you actually want to train or resume.

Core training controls live in `config_74_hidden45.yaml`, not in notebook cells:

```yaml
train:
  epochs: 12
  batch_size: 1
  grad_accum_steps: 16
  seed: 42
  checkpoint_interval: 1
  auto_resume: true
  resume: null
  eval_interval: 1
  eval_max_dets: 100
```

Auto-resume behavior:

- `auto_resume: true` + `resume: null` uses the latest `checkpoint_N.ckpt` in `output_dir`.
- Set `resume: /path/to/checkpoint_N.ckpt` to force a specific checkpoint.
- Set `auto_resume: false` and use a new `model.tag` or clean output folder for a fresh run.

For Colab/CUDA, update before training:

```yaml
train:
  device: cuda
  pin_memory: true
  num_workers: 2  # raise to 4 only if the runtime and Drive I/O are stable
output:
  local_output_dir: /content/drive/MyDrive/.../rfdetr_outputs
  backup_dir: /content/drive/MyDrive/.../rfdetr_outputs/best
```

Validation/evaluation notes:

- RF-DETR applies `seed: 42` at fit start with worker seeding, so reruns are seeded unless you change the config.
- RF-DETR evaluates `valid/_annotations.coco.json` during training because `eval_interval: 1` is set.
- The fast training metrics are written to `output_dir/metrics.csv`.
- The competition-facing check here is `mAP@0.75`; the wrapper also saves the best mAP75 epoch as `checkpoint_best_map75.ckpt` and `{model_tag}_best_map75.ckpt` after training completes.
- The notebook includes an optional RT-DETR-style custom validation pass for `mAP@[0.75:0.95]`.
- If a run was already started before this wrapper change, use `python train_74_hidden45.py --config config_74_hidden45.yaml --finalize-only` after it finishes to copy the best mAP75 checkpoint without retraining.


In [36]:
# Inspect training config
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
print(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))

data:
  dataset_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45
  base56_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/train_56_45_merged_coco
  hidden18_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/hidden_train_import
  test_images_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/sprint_ai_project1_data/test_images
model:
  variant: nano
  tag: nano_74_hidden45_mps
train:
  epochs: 12
  batch_size: 1
  grad_accum_steps: 16
  seed: 42
  lr: 0.0001
  lr_encoder: 0.00015
  weight_decay: 0.0001
  lr_scheduler: cosine
  warmup_epochs: 1.0
  lr_min_factor: 0.05
  checkpoint_interval: 1
  eval_interval: 1
  early_stopping: true
  early_stopping_patience: 10
  early_stopping_min_delta: 0.001
  early_stopping_use_ema: true
  use_ema: true
  tensorboard: true
  progress_bar: tqdm
  log_per_class_metrics: true
  run_test: false
  compute_val_loss: true
  compute_te

In [37]:
# Dry-run training entrypoint
if RUN_DRY_RUN:
    dry_run_cmd = [sys.executable, 'train_74_hidden45.py', '--config', str(CONFIG_PATH), '--dry-run']
    print('run:', ' '.join(dry_run_cmd))
    subprocess.run(dry_run_cmd, check=True)
else:
    print('Skipping dry-run.')

run: /Users/pio/.DataAnalysis/bin/python train_74_hidden45.py --config config_74_hidden45.yaml --dry-run


/Users/pio/.DataAnalysis/lib/python3.11/site-packages/rfdetr/models/weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, args_mapping={"train_config": None}, deprecated_in="1.7.0", remove_in="1.9.0", num_warns=-1)


{
  "dataset_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45",
  "output_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/nano_74_hidden45_mps",
  "backup_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/best",
  "model_variant": "nano",
  "model_tag": "nano_74_hidden45_mps",
  "train_kwargs": {
    "epochs": 12,
    "batch_size": 1,
    "grad_accum_steps": 16,
    "seed": 42,
    "lr": 0.0001,
    "lr_encoder": 0.00015,
    "weight_decay": 0.0001,
    "lr_scheduler": "cosine",
    "warmup_epochs": 1.0,
    "lr_min_factor": 0.05,
    "checkpoint_interval": 1,
    "eval_interval": 1,
    "early_stopping": true,
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.001,
    "early_stopping_use_ema": true,
    "use_ema": true,
    "tensorboard": true,
    "progress_bar": "tqdm",
    "log_per_class_metrics": true,
    "run_test": false,
    "compute_val_lo

In [38]:
# Full training / finalization
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
train_cfg = config['train']
print('config-controlled run settings:')
for key in ['epochs', 'batch_size', 'grad_accum_steps', 'lr', 'lr_encoder', 'lr_scheduler', 'checkpoint_interval', 'auto_resume', 'resume', 'eval_interval', 'eval_max_dets']:
    print(f'  {key}: {train_cfg.get(key)}')

train_cmd = [sys.executable, 'train_74_hidden45.py', '--config', str(CONFIG_PATH)]
if TRAIN_EPOCHS_OVERRIDE is not None:
    train_cmd += ['--epochs', str(TRAIN_EPOCHS_OVERRIDE)]
    print('WARNING: TRAIN_EPOCHS_OVERRIDE is active. Prefer config_74_hidden45.yaml for normal runs.')

finalize_cmd = [sys.executable, 'train_74_hidden45.py', '--config', str(CONFIG_PATH), '--finalize-only']
print('full train command:', ' '.join(train_cmd))
print('finalize-only command:', ' '.join(finalize_cmd))

if RUN_FINALIZE_ONLY:
    subprocess.run(finalize_cmd, check=True)
elif RUN_FULL_TRAIN:
    env = os.environ.copy()
    env.setdefault('WANDB_DISABLED', 'true')
    env.setdefault('WANDB_MODE', 'disabled')
    subprocess.run(train_cmd, check=True, env=env)
else:
    print('RUN_FULL_TRAIN=False and RUN_FINALIZE_ONLY=False, so no training/finalization was launched.')
    print('Set RUN_FULL_TRAIN=True to train/auto-resume, or RUN_FINALIZE_ONLY=True after a completed run to copy best checkpoints.')


full train command: /Users/pio/.DataAnalysis/bin/python train_74_hidden45.py --config config_74_hidden45.yaml


/Users/pio/.DataAnalysis/lib/python3.11/site-packages/rfdetr/models/weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, args_mapping={"train_config": None}, deprecated_in="1.7.0", remove_in="1.9.0", num_warns=-1)


{
  "dataset_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45",
  "output_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/nano_74_hidden45_mps",
  "backup_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/best",
  "model_variant": "nano",
  "model_tag": "nano_74_hidden45_mps",
  "train_kwargs": {
    "epochs": 12,
    "batch_size": 1,
    "grad_accum_steps": 16,
    "seed": 42,
    "lr": 0.0001,
    "lr_encoder": 0.00015,
    "weight_decay": 0.0001,
    "lr_scheduler": "cosine",
    "warmup_epochs": 1.0,
    "lr_min_factor": 0.05,
    "checkpoint_interval": 1,
    "eval_interval": 1,
    "early_stopping": true,
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.001,
    "early_stopping_use_ema": true,
    "use_ema": true,
    "tensorboard": true,
    "progress_bar": "tqdm",
    "log_per_class_metrics": true,
    "run_test": false,
    "compute_val_lo

[2026-07-03 20:25:37] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-03 20:25:37] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-07-03 20:25:38] [INFO] rf-detr - File /Users/pio/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


...

Metric __rfdetr_effective_map__ improved by 0.004 >= min_delta = 0.001. New best score: 0.972


[2026-07-03 22:44:31] [INFO] rf-detr - Best EMA mAP improved to 0.9715 (epoch 6)
Epoch 7: 100%|██████████| 3504/3504 [16:08<00:00,  3.62it/s, loss=1.620, loss_cls=0.142, loss_box=0.0301, loss_giou=0.0238, val/loss=1.280, val/mAP_50_95=0.968, val/mAP_50=0.976, val/ema_mAP_50_95=0.972, val/F1=0.946]   
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:  37%|███▋      | 140/376 [00:30<00:51,  4.56it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


KeyboardInterrupt: 

/opt/homebrew/Cellar/python@3.11/3.11.15_1/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/resource_tracker.py:254: UserWarning: resource_tracker: There appear to be 29 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '


In [ ]:
# Locate outputs/checkpoints
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
model_tag = config['model'].get('tag', config['model']['variant'])
output_dir = Path(config['output']['local_output_dir']) / model_tag
backup_dir = Path(config['output']['backup_dir'])

print('output_dir:', output_dir)
print('backup_dir:', backup_dir)
if output_dir.exists():
    for path in sorted(output_dir.glob('*')):
        print(path.name, f'{path.stat().st_size / (1024 * 1024):.1f} MB' if path.is_file() else '<dir>')
else:
    print('No output_dir yet. Run full training first.')

map75_summary_path = output_dir / 'map75_summary.json'
if map75_summary_path.exists():
    map75_summary = json.loads(map75_summary_path.read_text(encoding='utf-8'))
    print('mAP75 checkpoint summary:')
    display(pd.DataFrame([map75_summary]))
else:
    print('No map75_summary.json yet. It is written after a completed training run.')

if backup_dir.exists():
    best_candidates = sorted(backup_dir.glob(f'{model_tag}*'))
    print('best candidates:')
    for path in best_candidates:
        print('-', path, f'{path.stat().st_size / (1024 * 1024):.1f} MB')
else:
    print('No backup_dir yet.')


output_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/nano_74_hidden45_mps
backup_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/best
run_summary.json 0.0 MB
best candidates:


## Evaluation

Use two layers of validation:

1. Fast summary from RF-DETR's `metrics.csv` after training. This uses `val/mAP_75` as the primary model-selection metric for this project.
2. Optional RT-DETR-style custom validation inference. Set `RUN_VALIDATION_INFERENCE = True` after a checkpoint exists to compute `Custom mAP@[0.75:0.95]` with COCOeval over IoU thresholds `[0.75, 0.80, 0.85, 0.90, 0.95]`.

RF-DETR internally still writes its regular `checkpoint_best_total.pth` by `val/mAP_50_95`; the wrapper adds a separate `checkpoint_best_map75.ckpt` based on `val/mAP_75` after training completes.

The downloaded `test` split has no public annotations, so this notebook cannot compute true test mAP locally. Test inference/submission generation is a separate step.


In [ ]:
# Fast validation metrics from RF-DETR metrics.csv
metrics_csv = output_dir / 'metrics.csv'
print('metrics_csv:', metrics_csv)

if not metrics_csv.exists():
    print('metrics.csv not found yet. Run full training first, then rerun this section.')
else:
    metrics_df = pd.read_csv(metrics_csv)
    metric_cols = [
        'val/mAP_75',
        'val/mAP_50_95',
        'val/ema_mAP_50_95',
        'val/mAP_50',
        'val/mAR',
        'val/F1',
        'val/precision',
        'val/recall',
        'val/loss',
        'train/loss',
    ]
    available_cols = [col for col in metric_cols if col in metrics_df.columns]
    per_epoch = metrics_df.groupby('epoch')[available_cols].last().dropna(how='all').reset_index()

    rename = {
        'val/mAP_75': 'mAP@0.75',
        'val/mAP_50_95': 'mAP@0.50:0.95',
        'val/ema_mAP_50_95': 'EMA mAP@0.50:0.95',
        'val/mAP_50': 'mAP@0.50',
        'val/mAR': 'mAR',
        'val/F1': 'F1',
        'val/precision': 'precision',
        'val/recall': 'recall',
        'val/loss': 'val_loss',
        'train/loss': 'train_loss',
    }
    display(per_epoch.rename(columns=rename).tail(20))

    primary_col = 'val/mAP_75'
    if primary_col in per_epoch.columns and per_epoch[primary_col].notna().any():
        best_idx = per_epoch[primary_col].idxmax()
        best_row = per_epoch.loc[[best_idx]].rename(columns=rename)
        best_epoch = int(per_epoch.loc[best_idx, 'epoch'])
        best_score = float(per_epoch.loc[best_idx, primary_col])
        print(f'Best validation mAP@0.75: epoch={best_epoch}, score={best_score:.6f}')
        map75_ckpt = output_dir / f'checkpoint_{best_epoch}.ckpt'
        print('best mAP75 epoch checkpoint:', map75_ckpt, 'exists=' + str(map75_ckpt.exists()))
        print('post-train copied mAP75 checkpoint:', output_dir / 'checkpoint_best_map75.ckpt')
        display(best_row)
    else:
        print('val/mAP_75 is not present in metrics.csv yet.')

    ap_cols = [col for col in metrics_df.columns if col.startswith('val/AP/')]
    if ap_cols and primary_col in metrics_df.columns:
        valid_metric_rows = metrics_df[metrics_df[primary_col].notna()]
        if not valid_metric_rows.empty:
            best_metric_row = valid_metric_rows.loc[valid_metric_rows[primary_col].idxmax()]
            category_map_path = RFDETR_DATASET_DIR / 'category_mapping.csv'
            category_meta = pd.read_csv(category_map_path, encoding='utf-8-sig') if category_map_path.exists() else pd.DataFrame()
            meta_by_category = {}
            if not category_meta.empty:
                meta_by_category = category_meta.set_index('category_id').to_dict('index')

            per_class_rows = []
            for col in ap_cols:
                category_id = int(col.split('/')[-1])
                meta = meta_by_category.get(category_id, {})
                per_class_rows.append({
                    'category_id': category_id,
                    'n_number': meta.get('n_number', ''),
                    'name': meta.get('name', ''),
                    'AP': float(best_metric_row[col]) if pd.notna(best_metric_row[col]) else None,
                })
            per_class_df = pd.DataFrame(per_class_rows).sort_values('AP', na_position='last')
            print('Lowest per-class AP at the best mAP@0.75 epoch:')
            display(per_class_df.head(15))
            print('Highest per-class AP at the best mAP@0.75 epoch:')
            display(per_class_df.tail(15).sort_values('AP', ascending=False))


metrics_csv: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/nano_74_hidden45_mps/metrics.csv
metrics.csv not found yet. Run full training first, then rerun this section.


In [ ]:
# Plot RT-DETR-style training curves with mAP@0.75 highlighted
if not metrics_csv.exists():
    print('Skipping plot: metrics.csv not found yet.')
else:
    import matplotlib.pyplot as plt

    plot_df = metrics_df.groupby('epoch').last().reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    if 'train/loss' in plot_df.columns and plot_df['train/loss'].notna().any():
        axes[0].plot(plot_df['epoch'], plot_df['train/loss'], marker='o', label='train/loss')
    if 'val/loss' in plot_df.columns and plot_df['val/loss'].notna().any():
        axes[0].plot(plot_df['epoch'], plot_df['val/loss'], marker='o', label='val/loss')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].grid(True)
    axes[0].legend()

    metric_plot_cols = [
        ('val/mAP_75', 'mAP@0.75'),
        ('val/mAP_50_95', 'mAP@0.50:0.95'),
        ('val/mAP_50', 'mAP@0.50'),
        ('val/ema_mAP_50_95', 'EMA mAP@0.50:0.95'),
    ]
    for col, label in metric_plot_cols:
        if col in plot_df.columns and plot_df[col].notna().any():
            axes[1].plot(plot_df['epoch'], plot_df[col], marker='o', label=label)

    if 'val/mAP_75' in plot_df.columns and plot_df['val/mAP_75'].notna().any():
        best_idx = plot_df['val/mAP_75'].idxmax()
        axes[1].axvline(plot_df.loc[best_idx, 'epoch'], linestyle='--', color='gray', label='best mAP@0.75')

    axes[1].set_title('Validation mAP')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('mAP')
    axes[1].grid(True)
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


Skipping plot: metrics.csv not found yet.


In [ ]:
# Optional RT-DETR-style custom validation mAP@[0.75:0.95]
def _resolve_best_checkpoint(output_dir: Path) -> Path | None:
    candidates = [
        output_dir / 'checkpoint_best_total.pth',
        output_dir / 'checkpoint_best_ema.pth',
        output_dir / 'checkpoint_best_regular.pth',
    ]
    return next((path for path in candidates if path.exists()), None)

if not RUN_VALIDATION_INFERENCE:
    print('Skipping custom validation inference. Set RUN_VALIDATION_INFERENCE = True after full training to run it.')
else:
    from collections import Counter
    import numpy as np
    from PIL import Image
    from tqdm.auto import tqdm
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    from model import get_rfdetr_model

    checkpoint_path = _resolve_best_checkpoint(output_dir)
    if checkpoint_path is None:
        print('No best checkpoint found in output_dir yet:', output_dir)
    else:
        print('checkpoint:', checkpoint_path)
        val_json = RFDETR_DATASET_DIR / 'valid' / '_annotations.coco.json'
        val_dir = RFDETR_DATASET_DIR / 'valid'
        coco_gt = COCO(str(val_json))
        image_infos = list(coco_gt.dataset['images'])
        if VALIDATION_MAX_IMAGES is not None:
            image_infos = image_infos[:int(VALIDATION_MAX_IMAGES)]
            print(f'validation smoke subset: {len(image_infos)} images')

        category_map_path = RFDETR_DATASET_DIR / 'category_mapping.csv'
        category_map = pd.read_csv(category_map_path, encoding='utf-8-sig')
        valid_category_ids = set(int(c['id']) for c in coco_gt.dataset['categories'])
        internal_to_category = {
            int(row['rfdetr_internal_label']): int(row['category_id'])
            for _, row in category_map.iterrows()
        }

        def to_coco_category_id(raw_label: int) -> int | None:
            raw_label = int(raw_label)
            if raw_label in valid_category_ids:
                return raw_label
            if raw_label in internal_to_category:
                return internal_to_category[raw_label]
            # Some export paths are 1-based; keep this as a guard, but K-code IDs never overlap low internal labels.
            if (raw_label - 1) in internal_to_category:
                return internal_to_category[raw_label - 1]
            return None

        model_variant = config['model']['variant']
        model = get_rfdetr_model(model_variant, checkpoint_path=str(checkpoint_path))

        coco_results = []
        skipped_labels = Counter()
        for img_info in tqdm(image_infos, desc='val inference'):
            img_path = val_dir / img_info['file_name']
            image = np.array(Image.open(img_path).convert('RGB'))
            height, width = image.shape[:2]
            detections = model.predict(image, threshold=VALIDATION_SCORE_THR)

            boxes = np.asarray(detections.xyxy, dtype=float)
            scores = np.asarray(detections.confidence, dtype=float)
            labels = np.asarray(detections.class_id, dtype=int)
            if len(scores) == 0:
                continue

            order = np.argsort(scores)[::-1][:int(VALIDATION_MAX_DET_PER_IMAGE)]
            for idx in order:
                category_id = to_coco_category_id(labels[idx])
                if category_id is None:
                    skipped_labels[int(labels[idx])] += 1
                    continue

                x1, y1, x2, y2 = boxes[idx].tolist()
                x1 = max(0.0, min(float(x1), float(width)))
                y1 = max(0.0, min(float(y1), float(height)))
                x2 = max(0.0, min(float(x2), float(width)))
                y2 = max(0.0, min(float(y2), float(height)))
                if x2 <= x1 or y2 <= y1:
                    continue

                coco_results.append({
                    'image_id': int(img_info['id']),
                    'category_id': int(category_id),
                    'bbox': [x1, y1, x2 - x1, y2 - y1],
                    'score': float(scores[idx]),
                })

        print('num val predictions:', len(coco_results))
        if skipped_labels:
            print('skipped unknown raw labels:', dict(skipped_labels.most_common()))

        val_result_json = output_dir / 'val_predictions_map75_95.json'
        val_result_json.write_text(json.dumps(coco_results, ensure_ascii=False), encoding='utf-8')
        print('saved predictions:', val_result_json)

        if not coco_results:
            print('No predictions to evaluate.')
        else:
            coco_dt = coco_gt.loadRes(str(val_result_json))
            coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
            coco_eval.params.iouThrs = np.arange(0.75, 0.96, 0.05)
            coco_eval.params.maxDets = [1, 10, int(VALIDATION_MAX_DET_PER_IMAGE)]
            coco_eval.evaluate()
            coco_eval.accumulate()
            coco_eval.summarize()

            custom_metrics = {
                'checkpoint': str(checkpoint_path),
                'num_images': len(image_infos),
                'num_predictions': len(coco_results),
                'score_threshold': float(VALIDATION_SCORE_THR),
                'max_det_per_image': int(VALIDATION_MAX_DET_PER_IMAGE),
                'custom_mAP_75_95': float(coco_eval.stats[0]),
                'custom_AP_75': float(coco_eval.stats[2]),
                'custom_AR_75_95_max_det': float(coco_eval.stats[8]),
            }
            custom_metrics_json = output_dir / 'validation_custom_map75_95.json'
            custom_metrics_json.write_text(json.dumps(custom_metrics, ensure_ascii=False, indent=2), encoding='utf-8')
            display(pd.DataFrame([custom_metrics]))
            print('Custom mAP@[0.75:0.95]:', custom_metrics['custom_mAP_75_95'])
            print('Custom AP@0.75:', custom_metrics['custom_AP_75'])
            print('saved metrics:', custom_metrics_json)


Skipping custom validation inference. Set RUN_VALIDATION_INFERENCE = True after full training to run it.


## Checks

Expected safe-run result with default parameters:

- dataset preparation succeeds
- `train` has 3502 images and 3965 annotations
- `valid` has 376 images and 452 annotations
- `test` has 842 original test images and 0 annotations
- train+valid has no class below 45 annotations
- dry-run exits successfully

When `RUN_FULL_TRAIN=True`:

- config controls `epochs`, `batch_size`, `grad_accum_steps`, LR, checkpoint interval, and auto-resume
- `auto_resume: true` resumes from the latest `checkpoint_N.ckpt` if `resume` is null
- RF-DETR writes its regular best checkpoint as `checkpoint_best_total.pth`
- wrapper copies `checkpoint_best_total.pth` to `{model_tag}_best.pth`
- wrapper also copies the best `val/mAP_75` epoch checkpoint to `checkpoint_best_map75.ckpt` and `{model_tag}_best_map75.ckpt`
- `metrics.csv` exists in `output_dir`
- the Evaluation section reports best `val/mAP_75`
- optional custom validation inference reports RT-DETR-style `Custom mAP@[0.75:0.95]`

`RUN_FINALIZE_ONLY=True` skips training and only refreshes `{model_tag}_best.pth`, `checkpoint_best_map75.ckpt`, `{model_tag}_best_map75.ckpt`, and `map75_summary.json` from an existing completed output folder.
